# SER CNN — Experiment Progress Log
**Date:** 2026-07-01  
**Notebook:** new1.ipynb  
**Dataset:** uldisvalainis/audio-emotions (CREMA-D + RAVDESS only)  
**Task:** 6-class Speech Emotion Recognition — Angry, Disgusted, Fearful, Happy, Neutral, Sad  
**Setup:** SR=16000, duration filter 1.0–3.5s, seed=42, ~7343 clips → 5874 train / 1469 test

## Run 1 — Baseline
**Result: 0.634**

### Configuration
- **Channels (7):** mel, delta, delta2, chroma, rms, f0, rolloff
- **Model:** 3-block CNN → AdaptiveAvgPool(4,4) → Linear(2048, 128) → Linear(128, 6)
- **Loss:** CrossEntropyLoss + class weights (balanced)
- **Optimizer:** Adam lr=3e-4
- **Scheduler:** ReduceLROnPlateau (patience=5, factor=0.5, min_lr=1e-5)
- **Epochs:** 40
- **Regularization:** Dropout(0.4) in FC only

### Observations
- Large overfitting gap: train ~0.80 vs test 0.634 (gap ~0.161)
- ReduceLROnPlateau cut LR too early when test accuracy oscillated, stalling training
- Strong classes: Angry, Neutral. Weak: Disgusted, Fearful

## Run 2 — SpecAugment (did not help)
**Result: 0.623–0.631 (below baseline)**

### What we tried
- Added time masking and frequency masking to training data
- Tried multiple mask sizes: large (time 5–20, freq 5–25), then smaller (time 5–12, freq 5–15)

### Why it failed
- Large masks destroyed the subtle spectral patterns that distinguish Neutral — Neutral accuracy dropped significantly
- Even reduced masks couldn't recover the baseline
- Dataset is small (~5874 train clips); aggressive augmentation hurt more than it helped

### Decision
Dropped augmentation entirely. Reverted to clean baseline.

## Run 3 — Expanded Feature Set (11 channels)
**Result: 0.639**

### What we added
- **Spectral bandwidth:** energy spread around spectral centroid
- **Spectral contrast:** peak vs valley energy ratio per sub-band (7 bands → resized to 128)
- **Tempogram:** local tempo autocorrelation (128×T) — captures rhythmic patterns in speech
- Also added: pitch_conf (voiced probability from pyin)

### Why it helped
- More spectral texture information gave the CNN additional discriminative dimensions
- Contrast especially captures the harmonic structure differences between emotions

### Permutation feature importance (run on this model)
| Channel | Importance (acc drop) |
|---------|----------------------|
| rms | +0.199 ← #1 |
| delta2 | +0.167 |
| delta | +0.148 |
| f0 | +0.124 |
| mel | +0.105 |
| rolloff | +0.103 |
| contrast | +0.101 |
| bandwidth | +0.047 |
| chroma | ~0.010 |
| pitch_conf | ~0.008 |
| tempogram | ~0.005 |

### Decision
Dropped chroma, pitch_conf, tempogram (low importance). Kept 8 channels.

## Run 4 — SpatialDropout2d + Label Smoothing (8 channels)
**Result: 0.641**

### What we changed
- **SpatialDropout2d(0.15)** added after each conv block (drops entire feature maps, stronger regularization than pixel dropout)
- **Label smoothing = 0.1** in CrossEntropyLoss (softens hard targets, reduces overconfidence)
- Dropped chroma, pitch_conf, tempogram → 8 active channels
- Epochs: 100

### Why it helped
- SpatialDropout forces the model to not rely on any single feature map → better generalization
- Label smoothing prevents the model from becoming overconfident on training examples
- Together they cut the overfitting gap from 0.161 → 0.064

### Per-class results (at 0.641)
| Class | F1 |
|-------|----|
| Angry | 0.744 |
| Neutral | 0.696 |
| Sad | 0.640 |
| Happy | 0.614 |
| Fearful | 0.603 |
| Disgusted | 0.579 |

## Run 5 — CosineAnnealingLR, 150 epochs
**Result: 0.647 ← new best at the time**

### What we changed
- Replaced **ReduceLROnPlateau** with **CosineAnnealingLR(T_max=150, eta_min=1e-5)**
- Increased epochs: 100 → 150

### Why it helped
- ReduceLROnPlateau cuts LR reactively when validation loss plateaus — but noisy test accuracy triggered it too early, freezing training prematurely
- Cosine decay is smooth and predictable: LR decays gradually from 3e-4 to 1e-5 over the full run regardless of validation signal
- The fine-tuning phase (epochs 125–150, lr ~3e-5 → 1e-5) kept finding new peaks: 0.642 → 0.644 → **0.647**

### LR schedule
```
epoch   1  lr 3.00e-4  (full speed)
epoch  50  lr 2.28e-4
epoch 100  lr 8.25e-5  (fine-tuning)
epoch 150  lr 1.00e-5  (final polish)
```

## Run 6 — Focal Loss (did not help)
**Result: 0.638 (below 0.647 baseline)**

### What we tried
- Replaced CrossEntropyLoss with **FocalLoss(gamma=2.0, label_smoothing=0.05)**
- Focal loss down-weights easy examples, forcing model to focus on hard ones (Disgusted, Fearful)
- Reduced label_smoothing from 0.1 → 0.05

### Why it failed
- Class weights (balanced) were already handling the imbalance signal — focal loss was redundant
- Reducing label_smoothing from 0.1 → 0.05 likely hurt regularization
- gamma=2.0 may have suppressed too much gradient from medium-confidence examples
- Disgusted and Fearful did NOT improve; most other classes got slightly worse

### Per-class F1 comparison
| Class | Run 5 (0.647) | Run 6 focal (0.638) | Δ |
|-------|--------------|---------------------|---|
| Angry | 0.744 | 0.722 | -0.022 |
| Disgusted | 0.579 | 0.561 | -0.018 |
| Fearful | 0.603 | 0.597 | -0.006 |
| Happy | 0.614 | 0.597 | -0.017 |
| Neutral | 0.696 | 0.683 | -0.013 |
| Sad | 0.640 | 0.660 | +0.020 |

## Run 7 — Warm Restart on Focal Loss Model
**Result: 0.650 ← current best**

### What happened
- After 150 epochs of focal loss (best 0.638), ran 50 more epochs with EPOCHS=50 without restarting the model
- This reset the CosineAnnealingLR schedule back to lr=3e-4 → 1e-5 over 50 epochs (a warm restart)
- The high LR kick at epoch 1 allowed the model to escape the local minimum it was stuck in

### Why it worked
- Warm restarts (SGDR concept) periodically reset the LR to help the optimizer escape local minima
- The model already had good weights; the LR kick let it explore nearby better solutions
- Best appeared at epoch 45 (test 0.650) before the LR decayed to near-zero

### Per-class F1 at 0.650
| Class | F1 |
|-------|----|
| Angry | 0.743 |
| Neutral | 0.687 |
| Sad | 0.673 ← biggest gain (+0.033) |
| Happy | 0.610 |
| Fearful | 0.603 |
| Disgusted | 0.579 ← unchanged, structural bottleneck |

## Summary — What Worked vs What Didn't

| Change | Result | Verdict |
|--------|--------|---------|
| Baseline (7ch, ReduceLROnPlateau, 40ep) | 0.634 | starting point |
| SpecAugment | 0.623–0.631 | ✗ hurt Neutral, not enough data |
| +bandwidth, contrast, tempogram (11ch) | 0.639 | ✓ more spectral info helped |
| Drop low-importance channels → 8ch | — | ✓ cleaner signal |
| SpatialDropout2d(0.15) + label_smoothing=0.1 | 0.641 | ✓ gap 0.161→0.064 |
| CosineAnnealingLR + 150 epochs | 0.647 | ✓ smooth decay > reactive cuts |
| Focal loss (gamma=2.0) | 0.638 | ✗ redundant with class weights |
| Warm restart (50ep on focal model) | 0.650 | ✓ escapes local minima |
| **3-model ensemble (FC=128 + FC=256 + FC=128 s7)** | **0.654** | ✓ **current best** |
| Audio aug all classes + ensemble | 0.652 | ✗ hurt strong classes |
| Audio aug Disgusted+Fearful only + ensemble | in progress | — |

## Bottleneck Analysis

**Disgusted** (F1=0.577) is the structural ceiling.
- Acoustically close to Angry (both high-arousal, negative valence)
- No loss function, architecture, or ensemble change moved it — this is a **data limitation**
- CREMA-D + RAVDESS have limited actor diversity for Disgusted specifically
- RMS is the #1 feature (energy = arousal), but Disgusted and Angry share similar energy → harder to separate

## Active Channels (final 8)
```
mel, delta, delta2    — spectrogram + temporal dynamics
rms                   — energy envelope (#1 most important feature, drop=0.199)
f0                    — fundamental frequency / pitch
rolloff               — spectral brightness
bandwidth             — energy spread around centroid
contrast              — harmonic peak vs valley ratio
```

## Final Per-class Results (ensemble, 0.654)
| Class | Precision | Recall | F1 |
|-------|-----------|--------|-----|
| Angry | 0.739 | 0.769 | 0.754 |
| Disgusted | 0.600 | 0.555 | 0.577 |
| Fearful | 0.631 | 0.599 | 0.614 |
| Happy | 0.656 | 0.616 | 0.635 |
| Neutral | 0.643 | 0.739 | 0.688 |
| Sad | 0.641 | 0.649 | 0.645 |

## Known Gaps / To Improve

- **No early stopping in ensemble cell** — `train_model()` runs the full `EPOCHS=150` regardless of plateau. The best weights are captured via `best_state` so the final model is correct, but GPU time is wasted after the model stops improving. Should add a patience-based early stop (e.g. patience=20) to cut wasted epochs and allow faster iteration.


## Run 8 — 3-Model Ensemble (FC=128 s42 + FC=256 s42 + FC=128 s7)
**Result: 0.654 ← current best**

### What we did
- Trained 3 models independently, all with CE + label_smoothing=0.1 + CosineAnnealingLR 150ep:
  - **Model A:** FC=128, seed=42 → 0.650
  - **Model B:** FC=256, seed=42 → 0.649
  - **Model C:** FC=128, seed=7  → 0.642
- Averaged softmax probabilities across all 3 models before taking argmax

### Why it worked
- Each model makes different errors due to different random init and FC capacity
- Averaging softmax smooths out individual model biases — a prediction only wins if at least two models agree
- Diversity between models (different seeds + different architecture) is key; similar models don't add much

### Per-class F1 vs previous best
| Class | 0.650 (single) | 0.654 (ensemble) | Δ |
|-------|---------------|------------------|---|
| Angry | 0.743 | 0.754 | +0.011 |
| **Happy** | 0.610 | **0.635** | **+0.025** |
| Fearful | 0.603 | 0.614 | +0.011 |
| Neutral | 0.687 | 0.688 | ~0 |
| Sad | 0.673 | 0.645 | -0.028 |
| Disgusted | 0.579 | 0.577 | ~0 |

### Key observation
- Happy (+0.025) and Fearful (+0.011) benefited most — these were classes where individual models disagreed most, so averaging corrected the bias
- Sad dipped (-0.028) — the focal warm-restart model was unusually strong on Sad; averaging diluted that
- Disgusted unchanged — ensemble cannot fix a data-level ceiling

## Evaluation Methodology — Speaker-Independent Split

### The problem with random train/test split
All previous runs used `train_test_split(stratify=y)` which splits **clips randomly**. Since CREMA-D has 91 actors and RAVDESS has 24, the same speaker inevitably appears in both train and test. The model can partially learn **voice identity** (a speaker who sounds "angry" in training looks the same in test) rather than purely emotion — inflating reported accuracy.

### Why not train/val/test?
A three-way split was considered as an alternative to address checkpoint selection bias (using the test set for early stopping and best-model selection is technically leakage). The options were:

| Option | Pro | Con |
|--------|-----|-----|
| Random 70/15/15 | Unbiased final eval | Loses ~400 training clips; val set (~600) still noisy |
| Speaker-independent 80/20 | Eliminates speaker leakage; larger train set | Minor checkpoint selection bias remains |

**Decision: speaker-independent 80/20 split.**

Speaker leakage is the more serious validity threat — it can mask a model that is doing speaker recognition rather than emotion recognition. Checkpoint selection bias is a smaller effect and can be acknowledged in the report. Keeping 80% of speakers in train also preserves more data on a ~4k clip dataset.

### Implementation (cell-05-split)
- Speaker IDs extracted from filenames:
  - CREMA-D: first 4 digits (`1001_DFA_ANG.wav` → `CREMA_1001`)
  - RAVDESS: last field (`03-01-05-01-02-01-14.wav` → `RAVDESS_14`)
- ~115 unique speakers shuffled with `seed=42`
- 20% of speakers (~23) held out → all their clips go to test
- Emotion distribution printed at runtime to verify balance

### Expected impact
- Reported accuracy will likely **drop** vs previous runs — the model can no longer rely on speaker voice characteristics
- The new number is a more honest estimate of generalisation to unseen speakers
- This is the correct evaluation setup for a real-world SER deployment

## Run 9 — Architecture Overhaul + 4 New Features
**Status: in progress**

### Architecture changes (cell-06-model)
| Component | Before | After |
|-----------|--------|-------|
| Conv kernel | 3×3 only | 3×3 + 1×5 parallel (summed) |
| Channel recalibration | none | SE block per conv block |
| Skip connections | none | 1×1 conv + BN + pool skip |
| Depth | 3 blocks (32→64→128) | 4 blocks (32→64→128→256) |
| FC head input | 128×4×4 = 2048 | 256×4×4 = 4096 |
| Optimizer | Adam | AdamW (weight_decay=1e-2) |

**Why each change:**
- **Asymmetric 1×5 kernel** — spectrograms are not isotropic; time axis (prosody, rhythm) benefits from wider temporal receptive field than frequency axis
- **SE blocks** — with 12 heterogeneous channels (mel, rms, f0...) the model needs to learn which channels matter per input, not treat them equally
- **Residual skip** — 4 blocks deep; skip connections prevent gradient vanishing and allow the model to bypass unhelpful blocks
- **4th block (→256)** — more representational capacity in the final feature maps before pooling
- **AdamW** — decoupled weight decay regularises better than L2 via Adam momentum

### New features (cell-04-features → cell-c8209666)
| Feature | What it captures | Why added |
|---------|-----------------|-----------|
| `centroid` | spectral centre of mass | brightness correlates with arousal (angry/happy vs sad/neutral) |
| `onset` | energy flux / attack rate | fast onsets in angry, slow ramp in sad |
| `tempogram` | local tempo autocorrelation | speech rhythm and stress patterns |
| `voiced_prob` | per-frame voicing probability | speaking style, pauses, breathiness |

Total channels: 12 → 16 (mfcc, mfcc_delta, flatness, zcr still disabled)

### LR schedule change
Switched to **linear warmup (10 ep) + single cosine decay** after two failed attempts:
- `ReduceLROnPlateau(patience=8)` — dropped LR 5× by epoch 77, model idled at min_lr for 73 epochs
- `CosineAnnealingWarmRestarts(T_0=30)` — restart at epoch 90 spiked LR to 3e-4, destroying convergence (0.587 → 0.550)
- Warmup+cosine avoids both: no reactive cutting, no destructive restarts

### Early stopping setup
- `patience=25`, `grace=15` (no counting during warmup + early instability)
- Patience tracks **smoothed acc** (rolling mean of last 3 epochs) to avoid triggering on single noisy epochs
- Checkpoint saved on **raw best** so true peaks are captured